# 🧠 Entrenamiento de Redes Neuronales — MNIST

Este notebook permite entrenar y guardar **tres implementaciones diferentes** de redes neuronales para clasificar dígitos del dataset MNIST:

1. **BasicNeuralNetwork** — Red neuronal clásica con entrenamiento centralizado (full-batch gradient descent)
2. **ArnoviNeuralNetwork** — Entrenamiento distribuido con promediado de pesos al final
3. **DiegoNeuralNetwork** — Entrenamiento federado con sincronización de pesos por época

---

## Arquitectura común de las tres redes:
- **Entrada:** 784 neuronas (imagen 28×28 aplanada)
- **Capa oculta:** 128 neuronas (ReLU)
- **Salida:** 10 neuronas (Softmax — una por dígito 0-9)

## 📦 Importar módulos y cargar datos

In [ ]:
import numpy as np
import sys
import os

# Agregar el directorio actual al path
sys.path.insert(0, os.getcwd())

# Importar módulos del proyecto
from DatasetHandling import cargar_mnist, preprocesar
from Fuctions import forward, backward, cross_entropy, precision, predecir
from WeightsHandling import inicializar_pesos, actualizar_pesos
from ModelPersistence import guardar_modelo, cargar_modelo, probar_modelo, listar_modelos

print("✓ Módulos importados correctamente")

In [ ]:
# Cargar y preprocesar MNIST (esto se usará para los 3 entrenamientos)
print("Cargando dataset MNIST...")
X_all, y_all = cargar_mnist()

print("\nPreprocesando datos (aplanar, normalizar, one-hot, split 70/30)...")
X_train, Y_train, y_train, X_test, Y_test, y_test = preprocesar(X_all, y_all)

print("\n✓ Dataset listo para entrenar")
print(f"  - Train: {X_train.shape[0]} muestras")
print(f"  - Test:  {X_test.shape[0]} muestras")

---

# 1️⃣ BasicNeuralNetwork — Entrenamiento Centralizado

**Descripción:** Red neuronal clásica que entrena usando **todos los datos de entrenamiento** en cada época (Full-Batch Gradient Descent).

**Flujo:**
1. Inicializar pesos aleatorios
2. Repetir por E épocas:
   - Forward pass (calcular predicciones)
   - Calcular pérdida (cross-entropy)
   - Backward pass (calcular gradientes)
   - Actualizar pesos (gradient descent)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HIPERPARÁMETROS — BasicNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

BASIC_EPOCAS = 100       # Número de épocas de entrenamiento
BASIC_LR = 0.4           # Learning rate (tasa de aprendizaje)
BASIC_LOG_INTERVALO = 10 # Mostrar progreso cada N épocas

In [ ]:
def entrenar_basic(X_train, Y_train, y_train, X_test, y_test,
                   epocas=100, lr=0.3, intervalo_log=10):
    """
    Entrena una red neuronal con Full-Batch Gradient Descent.
    """
    print("\n" + "="*60)
    print("  INICIALIZANDO PESOS")
    print("="*60)
    W1, b1, W2, b2 = inicializar_pesos()

    historial_loss = []
    historial_acc  = []

    print("\n" + "="*60)
    print("  INICIANDO ENTRENAMIENTO")
    print(f"  Épocas: {epocas}  |  Learning Rate: {lr}")
    print(f"  Muestras en entrenamiento: {X_train.shape[0]}")
    print("="*60)

    for epoca in range(1, epocas + 1):
        # Forward Pass
        Z1, A1, Z2, A2 = forward(X_train, W1, b1, W2, b2)

        # Pérdida
        loss = cross_entropy(A2, Y_train)
        historial_loss.append(loss)

        # Precisión en entrenamiento
        y_pred_train = np.argmax(A2, axis=1)
        acc_train = precision(y_pred_train, y_train)
        historial_acc.append(acc_train)

        # Backward Pass
        dW1, db1, dW2, db2 = backward(X_train, Y_train, Z1, A1, A2, W2)

        # Actualizar Pesos
        W1, b1, W2, b2 = actualizar_pesos(W1, b1, W2, b2, dW1, db1, dW2, db2, lr)

        # Log de progreso
        if epoca % intervalo_log == 0 or epoca == 1:
            y_pred_test = predecir(X_test, W1, b1, W2, b2)
            acc_test = precision(y_pred_test, y_test)
            print(f"  Época {epoca:4d}/{epocas} │ "
                  f"Loss: {loss:.4f} │ "
                  f"Acc Train: {acc_train:.1f}% │ "
                  f"Acc Test: {acc_test:.1f}%")

    # Evaluación final
    print("\n" + "="*60)
    print("  EVALUACIÓN FINAL")
    print("="*60)
    y_pred_test = predecir(X_test, W1, b1, W2, b2)
    acc_final = precision(y_pred_test, y_test)
    print(f"\n  Precisión final en TEST: {acc_final:.2f}%")

    return W1, b1, W2, b2, historial_loss, historial_acc, acc_final

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ENTRENAR BasicNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  🧠 BASIC NEURAL NETWORK — ENTRENAMIENTO CENTRALIZADO")
print("="*60)

W1_basic, b1_basic, W2_basic, b2_basic, loss_basic, acc_basic, acc_final_basic = entrenar_basic(
    X_train, Y_train, y_train,
    X_test, y_test,
    epocas=BASIC_EPOCAS,
    lr=BASIC_LR,
    intervalo_log=BASIC_LOG_INTERVALO
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GUARDAR MODELO — BasicNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

guardar_modelo(
    W1_basic, b1_basic, W2_basic, b2_basic,
    nombre_modelo='BasicNN',
    precision_test=acc_final_basic,
    epocas=BASIC_EPOCAS,
    learning_rate=BASIC_LR
)

---

# 2️⃣ ArnoviNeuralNetwork — Entrenamiento Distribuido

**Descripción:** Divide el dataset en K particiones. Cada "worker" entrena su propia copia de la red con su partición. **Al final** se promedian los pesos de todas las redes.

**Flujo:**
1. Inicializar pesos compartidos
2. Dividir train en K particiones
3. Para cada partición: entrenar copia de la red por E épocas
4. Promediar los K conjuntos de pesos → modelo final

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HIPERPARÁMETROS — ArnoviNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

ARNOVI_PARTICIONES = 5       # Número de particiones (workers)
ARNOVI_EPOCAS = 100          # Épocas de entrenamiento por partición
ARNOVI_LR = 0.3              # Learning rate
ARNOVI_LOG_INTERVALO = 10    # Mostrar progreso cada N épocas

In [ ]:
def particionar_dataset(X_train, Y_train, y_train, num_particiones):
    """Divide el dataset en K particiones iguales (mezcladas aleatoriamente)."""
    N = X_train.shape[0]
    indices = np.random.permutation(N)
    X_mezclado = X_train[indices]
    Y_mezclado = Y_train[indices]
    y_mezclado = y_train[indices]

    X_partes = np.array_split(X_mezclado, num_particiones)
    Y_partes = np.array_split(Y_mezclado, num_particiones)
    y_partes = np.array_split(y_mezclado, num_particiones)

    particiones = []
    for k in range(num_particiones):
        particiones.append((X_partes[k], Y_partes[k], y_partes[k]))

    print(f"\n  Dataset dividido en {num_particiones} particiones:")
    for k, (X_k, Y_k, y_k) in enumerate(particiones):
        print(f"    Partición {k+1}: {X_k.shape[0]:5d} muestras")

    return particiones


def entrenar_particion(X_k, Y_k, y_k, W1_init, b1_init, W2_init, b2_init,
                       id_particion, epocas, lr, intervalo_log, X_test, y_test):
    """Entrena una red con una sola partición del dataset."""
    W1 = np.copy(W1_init)
    b1 = np.copy(b1_init)
    W2 = np.copy(W2_init)
    b2 = np.copy(b2_init)

    historial_loss = []
    historial_acc  = []

    print(f"\n  {'─'*55}")
    print(f"  PARTICIÓN {id_particion} — {X_k.shape[0]} muestras")
    print(f"  {'─'*55}")

    for epoca in range(1, epocas + 1):
        Z1, A1, Z2, A2 = forward(X_k, W1, b1, W2, b2)
        loss = cross_entropy(A2, Y_k)
        historial_loss.append(loss)
        y_pred_k = np.argmax(A2, axis=1)
        acc_k = precision(y_pred_k, y_k)
        historial_acc.append(acc_k)
        dW1, db1, dW2, db2 = backward(X_k, Y_k, Z1, A1, A2, W2)
        W1, b1, W2, b2 = actualizar_pesos(W1, b1, W2, b2, dW1, db1, dW2, db2, lr)

        if epoca % intervalo_log == 0 or epoca == 1:
            y_pred_test = predecir(X_test, W1, b1, W2, b2)
            acc_test = precision(y_pred_test, y_test)
            print(f"    [P{id_particion}] Época {epoca:4d}/{epocas} │ "
                  f"Loss: {loss:.4f} │ Acc Part: {acc_k:.1f}% │ Acc Test: {acc_test:.1f}%")

    return W1, b1, W2, b2, historial_loss, historial_acc


def promediar_pesos(lista_pesos):
    """Promedia los pesos de K redes entrenadas."""
    lista_W1 = [pesos[0] for pesos in lista_pesos]
    lista_b1 = [pesos[1] for pesos in lista_pesos]
    lista_W2 = [pesos[2] for pesos in lista_pesos]
    lista_b2 = [pesos[3] for pesos in lista_pesos]

    W1_prom = np.mean(np.array(lista_W1), axis=0)
    b1_prom = np.mean(np.array(lista_b1), axis=0)
    W2_prom = np.mean(np.array(lista_W2), axis=0)
    b2_prom = np.mean(np.array(lista_b2), axis=0)

    return W1_prom, b1_prom, W2_prom, b2_prom

In [ ]:
def entrenar_arnovi(X_train, Y_train, y_train, X_test, y_test,
                    num_particiones, epocas, lr, intervalo_log):
    """
    Algoritmo de Arnovi: entrenamiento distribuido con promediado al final.
    """
    print("\n" + "="*60)
    print("  PASO 1: INICIALIZANDO PESOS COMPARTIDOS")
    print("="*60)
    W1_init, b1_init, W2_init, b2_init = inicializar_pesos()

    print("\n" + "="*60)
    print("  PASO 2: PARTICIONANDO EL DATASET")
    print("="*60)
    particiones = particionar_dataset(X_train, Y_train, y_train, num_particiones)

    print("\n" + "="*60)
    print("  PASO 3: ENTRENANDO UNA RED POR PARTICIÓN")
    print("="*60)

    lista_pesos_entrenados = []

    for k, (X_k, Y_k, y_k) in enumerate(particiones):
        W1_k, b1_k, W2_k, b2_k, _, _ = entrenar_particion(
            X_k, Y_k, y_k,
            W1_init, b1_init, W2_init, b2_init,
            id_particion=k + 1,
            epocas=epocas,
            lr=lr,
            intervalo_log=intervalo_log,
            X_test=X_test,
            y_test=y_test
        )
        lista_pesos_entrenados.append((W1_k, b1_k, W2_k, b2_k))
        y_pred_k = predecir(X_test, W1_k, b1_k, W2_k, b2_k)
        acc_k = precision(y_pred_k, y_test)
        print(f"  → Partición {k+1} terminada │ Acc Test individual: {acc_k:.2f}%")

    print("\n" + "="*60)
    print("  PASO 4: PROMEDIANDO PESOS DE TODAS LAS REDES")
    print("="*60)
    W1_final, b1_final, W2_final, b2_final = promediar_pesos(lista_pesos_entrenados)
    print(f"  Pesos promediados de {num_particiones} redes")

    print("\n" + "="*60)
    print("  PASO 5: EVALUACIÓN DEL MODELO PROMEDIADO")
    print("="*60)
    y_pred_final = predecir(X_test, W1_final, b1_final, W2_final, b2_final)
    acc_final = precision(y_pred_final, y_test)
    print(f"\n  Precisión FINAL del modelo promediado en TEST: {acc_final:.2f}%")

    return W1_final, b1_final, W2_final, b2_final, acc_final

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ENTRENAR ArnoviNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  🧠 ARNOVI NEURAL NETWORK — ENTRENAMIENTO DISTRIBUIDO")
print("="*60)

W1_arnovi, b1_arnovi, W2_arnovi, b2_arnovi, acc_final_arnovi = entrenar_arnovi(
    X_train, Y_train, y_train,
    X_test, y_test,
    num_particiones=ARNOVI_PARTICIONES,
    epocas=ARNOVI_EPOCAS,
    lr=ARNOVI_LR,
    intervalo_log=ARNOVI_LOG_INTERVALO
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GUARDAR MODELO — ArnoviNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

guardar_modelo(
    W1_arnovi, b1_arnovi, W2_arnovi, b2_arnovi,
    nombre_modelo='ArnoviNN',
    precision_test=acc_final_arnovi,
    epocas=ARNOVI_EPOCAS,
    learning_rate=ARNOVI_LR,
    info_extra={'num_particiones': ARNOVI_PARTICIONES}
)

---

# 3️⃣ DiegoNeuralNetwork — Entrenamiento Federado

**Descripción:** Similar a Arnovi, pero los pesos se promedian **después de cada época**, no solo al final. Esto permite una sincronización más frecuente entre los "workers".

**Flujo:**
1. Inicializar pesos globales
2. Dividir train en K particiones fijas
3. Para cada época:
   - Cada partición hace UN paso de entrenamiento
   - Promediar los K conjuntos de pesos → nuevos pesos globales
4. Resultado: modelo con sincronización constante

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HIPERPARÁMETROS — DiegoNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

DIEGO_PARTICIONES = 5       # Número de particiones (workers)
DIEGO_EPOCAS = 100          # Épocas totales (rondas de sincronización)
DIEGO_LR = 0.3              # Learning rate
DIEGO_LOG_INTERVALO = 10    # Mostrar progreso cada N épocas

In [ ]:
def train_on_batch(X_k, Y_k, W1, b1, W2, b2, lr):
    """Entrena la red con UN SOLO batch (una pasada forward+backward+update)."""
    W1_local = np.copy(W1)
    b1_local = np.copy(b1)
    W2_local = np.copy(W2)
    b2_local = np.copy(b2)

    Z1, A1, Z2, A2 = forward(X_k, W1_local, b1_local, W2_local, b2_local)
    dW1, db1, dW2, db2 = backward(X_k, Y_k, Z1, A1, A2, W2_local)
    W1_local, b1_local, W2_local, b2_local = actualizar_pesos(
        W1_local, b1_local, W2_local, b2_local, dW1, db1, dW2, db2, lr
    )

    return W1_local, b1_local, W2_local, b2_local

In [ ]:
def entrenar_diego(X_train, Y_train, y_train, X_test, y_test,
                   num_particiones, epocas, lr, intervalo_log):
    """
    Algoritmo de Diego: entrenamiento federado con promediado por época.
    """
    print("\n" + "="*60)
    print("  PASO 1: INICIALIZANDO PESOS GLOBALES")
    print("="*60)
    W1, b1, W2, b2 = inicializar_pesos()

    print("\n" + "="*60)
    print("  PASO 2: PARTICIONANDO EL DATASET")
    print("="*60)
    particiones = particionar_dataset(X_train, Y_train, y_train, num_particiones)

    print("\n" + "="*60)
    print("  PASO 3: ENTRENAMIENTO FEDERADO (PROMEDIADO POR ÉPOCA)")
    print("="*60)

    historial_loss = []
    historial_acc  = []

    for epoca in range(1, epocas + 1):
        # Cada partición hace UN paso con los pesos globales actuales
        lista_pesos_epoca = []
        for k, (X_k, Y_k, y_k) in enumerate(particiones):
            W1_k, b1_k, W2_k, b2_k = train_on_batch(X_k, Y_k, W1, b1, W2, b2, lr)
            lista_pesos_epoca.append((W1_k, b1_k, W2_k, b2_k))

        # Promediar pesos de todas las particiones
        W1, b1, W2, b2 = promediar_pesos(lista_pesos_epoca)

        # Evaluar modelo promediado
        Z1_all, A1_all, Z2_all, A2_all = forward(X_train, W1, b1, W2, b2)
        loss = cross_entropy(A2_all, Y_train)
        historial_loss.append(loss)

        y_pred_train = np.argmax(A2_all, axis=1)
        acc_train = precision(y_pred_train, y_train)
        historial_acc.append(acc_train)

        if epoca % intervalo_log == 0 or epoca == 1:
            y_pred_test = predecir(X_test, W1, b1, W2, b2)
            acc_test = precision(y_pred_test, y_test)
            print(f"  Época {epoca:4d}/{epocas} │ "
                  f"Loss: {loss:.4f} │ "
                  f"Acc Train: {acc_train:.1f}% │ "
                  f"Acc Test: {acc_test:.1f}%")

    # Evaluación final
    print("\n" + "="*60)
    print("  EVALUACIÓN FINAL")
    print("="*60)
    y_pred_test = predecir(X_test, W1, b1, W2, b2)
    acc_final = precision(y_pred_test, y_test)
    print(f"\n  Precisión FINAL del modelo Diego en TEST: {acc_final:.2f}%")

    return W1, b1, W2, b2, acc_final

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ENTRENAR DiegoNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  🧠 DIEGO NEURAL NETWORK — ENTRENAMIENTO FEDERADO")
print("="*60)

W1_diego, b1_diego, W2_diego, b2_diego, acc_final_diego = entrenar_diego(
    X_train, Y_train, y_train,
    X_test, y_test,
    num_particiones=DIEGO_PARTICIONES,
    epocas=DIEGO_EPOCAS,
    lr=DIEGO_LR,
    intervalo_log=DIEGO_LOG_INTERVALO
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GUARDAR MODELO — DiegoNeuralNetwork
# ═══════════════════════════════════════════════════════════════════════════

guardar_modelo(
    W1_diego, b1_diego, W2_diego, b2_diego,
    nombre_modelo='DiegoNN',
    precision_test=acc_final_diego,
    epocas=DIEGO_EPOCAS,
    learning_rate=DIEGO_LR,
    info_extra={'num_particiones': DIEGO_PARTICIONES}
)

---

# 📊 Resumen de Modelos Entrenados

In [ ]:
# Listar todos los modelos guardados
modelos = listar_modelos()

In [ ]:
# Comparación de precisiones finales
print("\n" + "="*60)
print("  📈 COMPARACIÓN DE PRECISIONES FINALES")
print("="*60)
print(f"  BasicNN  : {acc_final_basic:.2f}%")
print(f"  ArnoviNN : {acc_final_arnovi:.2f}%")
print(f"  DiegoNN  : {acc_final_diego:.2f}%")
print("="*60)